# Agent IA

**Objectif** : construire un agent qui peut **utiliser des outils** pour répondre à des questions qu'un LLM seul ne saurait pas traiter.

## Un LLM seul ne peut pas…

Essayez de demander à ChatGPT :
- *"Quelle température fait-il à Paris **maintenant** ?"* → il n'a pas accès au temps réel
- *"Combien fait 4852 × 37.6 ?"* → il devine, souvent faux
- *"Quelle est l'actualité d'aujourd'hui ?"* → connaissances figées à sa date d'entraînement

**Un LLM, c'est puissant pour raisonner, mais il ne sait pas agir dans le monde réel.**

## Un agent = LLM + outils + boucle de décision

Un **agent**, c'est un LLM qu'on connecte à des **outils** (fonctions Python, APIs…). Le LLM **décide** lui-même quel outil appeler selon la question.

```
Question utilisateur
       ↓
┌──────────────────────┐
│ LLM : je réfléchis │ ← "Reason"
│ Quel outil utiliser ? │
└──────────────────────┘
       ↓
┌──────────────────────┐
│ Appel d'outil      │ ← "Act"
│ get_weather("Paris")  │
└──────────────────────┘
       ↓
┌──────────────────────┐
│ Résultat observé   │ ← "Observe"
│ "18°C, nuageux"       │
└──────────────────────┘
       ↓
   (peut reboucler)
       ↓
┌──────────────────────┐
│ Réponse finale     │
└──────────────────────┘
```

---
## Étape 0 — Setup

In [ ]:
!pip install -q openai requests

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ Clé API configurée")

✅ Clé API configurée


---
## Étape 1 — Créer les outils (fonctions Python)

Un outil = juste une **fonction Python normale**.

### Outil 1 : `get_weather` — API Open-Meteo (gratuite, sans clé)

In [ ]:
import requests

def get_weather(ville: str) -> str:
    """Récupère la météo actuelle d'une ville via Open-Meteo."""

    # Étape 1 : convertir le nom de la ville en coordonnées (géocodage)
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": ville, "count": 1, "language": "fr"}
    ).json()

    if not geo.get("results"):
        return f"Ville '{ville}' introuvable"

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    nom = geo["results"][0]["name"]
    pays = geo["results"][0].get("country", "")

    # Étape 2 : récupérer la météo
    meteo = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"
        }
    ).json()

    temp = meteo["current"]["temperature_2m"]
    vent = meteo["current"]["wind_speed_10m"]
    humidite = meteo["current"]["relative_humidity_2m"]

    return f"Météo à {nom} ({pays}) : {temp}°C, vent {vent} km/h, humidité {humidite}%"


# Test en direct
print(get_weather("Paris"))
print(get_weather("Tokyo"))

Météo à Paris (France) : 15.5°C, vent 10.9 km/h, humidité 37%
Météo à Tokyo (Japon) : 11.5°C, vent 2.2 km/h, humidité 86%


### Outil 2 : `calculator` — évaluer une expression mathématique

In [ ]:
def calculator(expression: str) -> str:
    """Évalue une expression mathématique. Ex : '(23 + 17) * 2'"""
    try:
        # eval() normalement dangereux, mais on restreint aux maths de base
        # Pour un vrai agent en prod, utiliser ast.literal_eval ou un parseur dédié
        autorise = "0123456789+-*/(). "
        if not all(c in autorise for c in expression):
            return "Erreur : caractères non autorisés"
        resultat = eval(expression)
        return f"Résultat : {resultat}"
    except Exception as e:
        return f"Erreur de calcul : {e}"


# Test
print(calculator("(23 + 17) * 2"))
print(calculator("125.5 / 3"))

Résultat : 80
Résultat : 41.833333333333336


---
## Étape 2 — Décrire les outils au LLM

Pour que le LLM sache **quels outils existent** et **comment les utiliser**, il faut les lui décrire dans un format standardisé.

C'est comme une **documentation** pour le LLM : il lit la description, les paramètres, et décide quand l'appeler.

In [ ]:
tools_description = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Récupère la météo actuelle d\'une ville (température, vent, humidité). À utiliser pour toute question sur le temps qu\'il fait.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ville": {
                        "type": "string",
                        "description": "Le nom de la ville (ex: Paris, New York, Tokyo)"
                    }
                },
                "required": ["ville"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Évalue une expression mathématique. À utiliser pour tout calcul numérique.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "L\'expression à évaluer, ex: '(23 + 17) * 2' ou '125.5 / 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# Dictionnaire pour appeler les fonctions par leur nom
tools_mapping = {
    "get_weather": get_weather,
    "calculator": calculator
}

print(f"✅ {len(tools_description)} outils déclarés")
for t in tools_description:
    print(f"   - {t['function']['name']} : {t['function']['description'][:60]}...")

✅ 2 outils déclarés
   - get_weather : Récupère la météo actuelle d'une ville (température, vent, h...
   - calculator : Évalue une expression mathématique. À utiliser pour tout cal...


**Point-clé** : c'est **le champ `description`** qui permet au LLM de savoir **quand** utiliser l'outil. Plus la description est claire, mieux l'agent fonctionne.

*Mauvaise description* : `"fonction météo"` → trop vague  
*Bonne description* : `"Récupère la météo actuelle d'une ville (température, vent, humidité). À utiliser pour toute question sur le temps qu'il fait."` → le LLM sait exactement quand s'en servir.

---
## Étape 3 — Premier appel : voir le LLM **décider** d'utiliser un outil

On va envoyer une question au LLM **avec** la liste des outils disponibles. Deux choses peuvent arriver :
1. Le LLM répond directement (pas besoin d'outil)
2. Le LLM renvoie une **demande d'appel d'outil** (`tool_call`)

In [ ]:
from openai import OpenAI

client = OpenAI()

question = "Quelle température fait-il à Paris ?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}],
    tools=tools_description  # ← on lui donne les outils !
)

message = response.choices[0].message

print(f"❓ Question : {question}\n")
print(f"📨 Réponse brute du LLM :")
print(f"   content : {message.content}")
print(f"   tool_calls : {message.tool_calls}")

❓ Question : Quelle température fait-il à Paris ?

📨 Réponse brute du LLM :
   content : None
   tool_calls : [ChatCompletionMessageFunctionToolCall(id='call_XVNlWTzDv6NHQkddJn0tdpfV', function=Function(arguments='{"ville":"Paris"}', name='get_weather'), type='function')]


**Observez** : `content` est `None`, mais `tool_calls` contient une demande d'appel à `get_weather` avec `{"ville": "Paris"}`.

**Le LLM n'a pas répondu, il a demandé à appeler un outil.** C'est nous (notre code) qui devons **exécuter** l'outil et lui renvoyer le résultat.

C'est ça, le pattern agent : le LLM **décide**, notre code **exécute**.

---
## Étape 4 — Construire la boucle complète

Le pattern complet :
1. Envoyer la question au LLM
2. Si le LLM demande un outil → l'exécuter et lui renvoyer le résultat
3. Répéter jusqu'à ce qu'il donne une réponse finale

On va faire tout ça avec des **logs visuels** pour voir ce qui se passe.

In [ ]:
import json

def agent(question, verbose=True):
    """Agent complet avec boucle ReAct."""

    messages = [{"role": "user", "content": question}]

    if verbose:
        print(f"❓ Question : {question}\n")

    # Boucle : le LLM peut enchaîner plusieurs appels d'outils
    for iteration in range(5):  # max 5 itérations pour éviter les boucles infinies

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_description
        )

        message = response.choices[0].message
        messages.append(message)  # on garde l'historique complet

        # Cas 1 : le LLM donne une réponse finale
        if not message.tool_calls:
            if verbose:
                print(f" Réponse finale : {message.content}")
            return message.content

        # Cas 2 : le LLM demande d'appeler un ou plusieurs outils
        for tool_call in message.tool_calls:
            nom_outil = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if verbose:
                print(f" Le LLM veut appeler : {nom_outil}({arguments})")

            # Exécuter l'outil
            fonction = tools_mapping[nom_outil]
            resultat = fonction(**arguments)

            if verbose:
                print(f" Résultat : {resultat}\n")

            # Renvoyer le résultat au LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": resultat
            })

    return "Trop d\'itérations, abandon."


# Premier test
agent("Quelle température fait-il à Paris ?")

❓ Question : Quelle température fait-il à Paris ?

 Le LLM veut appeler : get_weather({'ville': 'Paris'})
 Résultat : Météo à Paris (France) : 15.5°C, vent 10.9 km/h, humidité 37%

 Réponse finale : La température à Paris est de 15.5°C, avec un vent de 10.9 km/h et une humidité de 37%.


'La température à Paris est de 15.5°C, avec un vent de 10.9 km/h et une humidité de 37%.'

---
## Étape 5 — Parfois, l'agent n'utilise pas d'outil

Si la question peut se répondre sans outil, l'agent répond directement.

In [ ]:
agent("Qui a écrit Les Misérables ?")

❓ Question : Qui a écrit Les Misérables ?

 Réponse finale : Les Misérables a été écrit par Victor Hugo. C'est l'un de ses romans les plus célèbres, publié pour la première fois en 1862.


"Les Misérables a été écrit par Victor Hugo. C'est l'un de ses romans les plus célèbres, publié pour la première fois en 1862."

**👀 Aucun outil appelé** — le LLM savait répondre tout seul avec ses connaissances. C'est le LLM qui **décide** si un outil est nécessaire.

---
## 🔗 Étape 6 — Questions qui nécessitent **plusieurs outils enchaînés**

C'est ici que ça devient vraiment impressionnant. L'agent va enchaîner les outils tout seul, dans le bon ordre.

In [ ]:
agent("Quelle température fait-il à Tokyo, et combien ça fait en Fahrenheit ? (formule : °F = °C × 9/5 + 32)")

❓ Question : Quelle température fait-il à Tokyo, et combien ça fait en Fahrenheit ? (formule : °F = °C × 9/5 + 32)

 Le LLM veut appeler : get_weather({'ville': 'Tokyo'})
 Résultat : Météo à Tokyo (Japon) : 11.5°C, vent 2.2 km/h, humidité 86%

 Le LLM veut appeler : calculator({'expression': '11.5 * 9/5 + 32'})
 Résultat : Résultat : 52.7

 Réponse finale : Il fait actuellement 11.5°C à Tokyo, ce qui correspond à environ 52.7°F.


'Il fait actuellement 11.5°C à Tokyo, ce qui correspond à environ 52.7°F.'

**👀 Regardez bien :**
1. L'agent appelle d'abord `get_weather("Tokyo")` → obtient la température en °C
2. Puis il appelle `calculator` avec la bonne formule en remplaçant la valeur
3. Enfin il formule la réponse finale

**Il a raisonné sur l'ordre des étapes tout seul.** Personne ne lui a dit "d'abord la météo, puis le calcul". C'est ça, la magie des agents.

In [ ]:
# Un exemple encore plus impressionnant
agent("Je prévois un voyage à Rome, Madrid et Berlin. Dans quelle ville fait-il le plus chaud en ce moment ? Et quelle est la différence avec la plus froide ?")

❓ Question : Je prévois un voyage à Rome, Madrid et Berlin. Dans quelle ville fait-il le plus chaud en ce moment ? Et quelle est la différence avec la plus froide ?

 Le LLM veut appeler : get_weather({'ville': 'Rome'})
 Résultat : Météo à Rome (Italie) : 15.1°C, vent 10.5 km/h, humidité 51%

 Le LLM veut appeler : get_weather({'ville': 'Madrid'})
 Résultat : Météo à Madrid (Espagne) : 19.3°C, vent 2.8 km/h, humidité 45%

 Le LLM veut appeler : get_weather({'ville': 'Berlin'})
 Résultat : Météo à Berlin (Allemagne) : 12.8°C, vent 8.2 km/h, humidité 39%

 Le LLM veut appeler : calculator({'expression': '19.3 - 12.8'})
 Résultat : Résultat : 6.5

 Réponse finale : Actuellement, Madrid est la ville la plus chaude avec une température de 19.3°C, tandis que Berlin est la plus froide à 12.8°C. La différence de température entre Madrid et Berlin est de 6.5°C.


'Actuellement, Madrid est la ville la plus chaude avec une température de 19.3°C, tandis que Berlin est la plus froide à 12.8°C. La différence de température entre Madrid et Berlin est de 6.5°C.'

**L'agent :**
1. Appelle `get_weather` **3 fois** (Rome, Madrid, Berlin)
2. Compare les températures
3. Utilise `calculator` pour la différence
4. Donne une réponse complète

**Imaginez le pouvoir** : on a codé ~50 lignes, et on a un assistant qui peut enchaîner des raisonnements complexes.

---
## Étape 7 — À vous de jouer

### Exercice 1 : posez 5 questions variées à l'agent

Testez :
- Une question où il doit utiliser `get_weather`
- Une question où il doit utiliser `calculator`
- Une question sans outil
- Une question où il doit enchaîner les 2 outils
- Une question volontairement tordue pour le piéger

In [ ]:
# À vous !
agent("???")